In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
sys.path.insert(0, '../src')

from recsys.data import load_ratings, make_positive, temporal_split, user_items
from recsys.models import PopularModel, ItemKNN
from recsys.metrics import evaluate

ratings = load_ratings('../data')
pos = make_positive(ratings)
train, test = temporal_split(pos)
truth = user_items(test)
users = list(truth)

models = {
    'Popular': PopularModel(),
    'ItemKNN': ItemKNN(),
}

for name, model in models.items():
    start = time.time()
    model.fit(train)
    recs = model.recommend(users, k=50)
    print(f'{name}  ({time.time() - start:.1f} сек)')
    for k in [10, 20, 50]:
        print('  ', evaluate(recs, truth, k))

Popular  (0.2 сек)
   {'recall@10': 0.042648219581247405, 'ndcg@10': 0.07913936652165218}
   {'recall@20': 0.08145251724242082, 'ndcg@20': 0.08686627364132175}
   {'recall@50': 0.15499870462422113, 'ndcg@50': 0.10946566329753124}
ItemKNN  (1.6 сек)
   {'recall@10': 0.06659914870581359, 'ndcg@10': 0.10443951602538894}
   {'recall@20': 0.11140003893622212, 'ndcg@20': 0.11384667705757714}
   {'recall@50': 0.21311636597979902, 'ndcg@50': 0.1473315026996469}


In [2]:
from recsys.metrics import coverage, mean_popularity

n_items = train['item_id'].nunique()
item_counts = train['item_id'].value_counts().to_dict()

for name, model in models.items():
    recs = model.recommend(users, k=50)
    print(name)
    for k in [10, 50]:
        print(f'  k={k}: охват каталога {coverage(recs, n_items, k):.1%}, '
              f'средняя популярность рекомендаций {mean_popularity(recs, item_counts, k):.0f} оценок')

print('Средняя популярность фильма в каталоге:', round(sum(item_counts.values()) / n_items), 'оценок')

Popular
  k=10: охват каталога 2.5%, средняя популярность рекомендаций 1952 оценок
  k=50: охват каталога 7.3%, средняя популярность рекомендаций 1364 оценок
ItemKNN
  k=10: охват каталога 11.7%, средняя популярность рекомендаций 1581 оценок
  k=50: охват каталога 23.9%, средняя популярность рекомендаций 1192 оценок
Средняя популярность фильма в каталоге: 132 оценок
